## Workflow Testing

In [29]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd
import urllib3


notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import scripts.data_pull_functions 
import scripts.gather_historic_data 
import scripts.gather_data_to_forecast 
import scripts.model_training_functions 
import scripts.create_output_functions 

load_dotenv()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [30]:
import importlib

importlib.reload(scripts.data_pull_functions)
importlib.reload(scripts.gather_historic_data)
importlib.reload(scripts.gather_data_to_forecast)
importlib.reload(scripts.model_training_functions)
importlib.reload(scripts.create_output_functions)

from scripts.data_pull_functions import login_google_cloud
from scripts.gather_historic_data import gather_historic_data
from scripts.gather_data_to_forecast import gather_data_to_forecast
from scripts.model_training_functions import fit_xgboost_clarissa_version, generate_feed_forward_forecast
from scripts.create_output_functions import create_nbpl_file, create_next_day_gas_burn_file

In [ ]:
print('starting main.py')
client = login_google_cloud(project_name="bepc-prj-energy-prod")

print('login successful')

# Initializing dates
today_mdy = datetime.today().strftime("%m-%d-%Y")
today_ymd = datetime.today().strftime("%Y-%m-%d")
test_start_date = (datetime.today() - pd.Timedelta(value=8, unit='D')).strftime("%m-%d-%Y")
forecast_horizon_mdy = (datetime.today() + pd.Timedelta(value=9, unit='D')).strftime("%m-%d-%Y")
nbpl_date_mdy = (datetime.today() + pd.Timedelta(value=1, unit='D')).strftime("%m-%d-%Y")

lead_columns = ['datetime', 'site', 'year', 'month', 'day', 'hour', 'hour_end', 'day_of_week', 'gas_day', 'hourly_gas_burn_MMBtu', 'daily_gas_burn_MMBtu']


starting main.py
login successful


In [4]:
model_df = gather_historic_data(
    start='2023-01-01', 
    end=today_ymd, #Exclusive end so date specified will not be included. For most purposes end should be today's date
    client=client, 
    lead_columns=lead_columns, 
    yes_username=os.getenv('YES_USERNAME'), 
    yes_password = os.getenv('YES_PASSWORD'), 
    save_output=True
    )

running gather_historic_data.py
Gathering historic data...	The historic dataframe is saved to ./data/processed-data/historic_data_df.csv


In [5]:
forward_df = gather_data_to_forecast(
    forecast_start=today_ymd, 
    forecast_end=forecast_horizon_mdy, #Exclusive end so date specified will not be included.
    yes_username=os.getenv('YES_USERNAME'), 
    yes_password=os.getenv('YES_PASSWORD'), 
    save_output=True
    )

The forecasting dataframe is saved to ./data/processed-data/data_to_forecast_df.csv


In [ ]:
features = ["availability_mw", "load_forecast", "net_load_forecast", "wind_forecast", "temperature_forecast", "wind_speed_forecast", "total_offline_forecast", 
        "offline_ng_forecast", "offline_coal_forecast", "hour", "day_of_week", "month", "gas_lag_1", "gas_lag_24", "gas_lag_168", "gas_roll_24", "gas_roll_168"]


# test_start_date is just 8 days ago from the today's date
fit_results = fit_xgboost_clarissa_version(
    model_df, 
    sites=model_df['site'].unique(), 
    test_start_date=test_start_date, 
    features=features, 
    target = 'hourly_gas_burn_MMBtu', 
    save_model_file=f"G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Models/models {today_ymd}.joblib")

The model details are saved to G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Models/models 2026-09-03.joblib


In [23]:
predictions = generate_feed_forward_forecast(
    historic_df=model_df, 
    forward_df=forward_df, 
    models=fit_results['models'], 
    features=features, 
    save_output=True, 
    save_file=f"G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Forecasts/Hourly Gas Burn by Site Forecasts {today_ymd}.csv"
    )


A file containing predictions has been saved to G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Forecasts/Hourly Gas Burn by Site Forecasts 2026-09-03.csv


In [25]:
predictions.shape

(960, 8)

In [ ]:
create_nbpl_file(
    predictions=predictions, 
    template_file='G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/NBPL Submission Files/NBPL template no links.xlsx', 
    file_date=nbpl_date_mdy, 
    save_file=f"G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/NBPL Submission Files/NBPL Forecast {today_ymd}.xlsm"
    )

A file with the NBPL hourly forecasts is saved to G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\NBPL Submission Files\NBPL Forecast 2026-09-03.xlsm


In [31]:
create_next_day_gas_burn_file(
    predictions=predictions, 
    template_file="G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Next Day Gas Burn Files/Next Day Gas Burn Template.xlsx", 
    save_file=f"G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Next Day Gas Burn Files/Next Day Gas Burn {today_ymd}.xlsx"
    )

A file with the next day gas burns is saved to G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Next Day Gas Burn Files\Next Day Gas Burn 2026-09-03.xlsx


'08-26-2026'